# Experiment

In [ ]:
# Load your dataset
df = spark.read.format("csv").option("header", "true").load("your_data.csv")

# Convert timestamp to day of week
df = df.withColumn('day_of_week', dayofweek(df['trip_start_timestamp']))

# Split the data into training and test sets
train_data, test_data = df.randomSplit([0.7, 0.3])

# Define stages of the pipeline
indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_index")
encoder = OneHotEncoder(inputCol="day_of_week_index", outputCol="day_of_week_vec")
vectorAssembler = VectorAssembler(inputCols=['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'day_of_week_vec'], outputCol="features")
rf = RandomForestRegressor(featuresCol="features", labelCol="trip_duration")

# Setup the pipeline
stages = [indexer, encoder, vectorAssembler, rf]
pipeline = Pipeline(stages = stages)

# Train the model on the training data
model = pipeline.fit(train_data)

# Now we can use the model to predict the trip duration in the test set
predictions = model.transform(test_data)

# Show some prediction results
predictions.select('trip_duration', 'prediction').show()
